In [ ]:
!pip install pandas
!pip install seaborn
!pip install matplotlib
!pip install statsmodels
!pip install sklearn

import pandas as pd  # data tables
import numpy as np  # math helpers
from sklearn.model_selection import train_test_split  # splits data into train/test
from sklearn.preprocessing import StandardScaler  # puts all features on the same scale
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV  # the models
from sklearn.metrics import r2_score, mean_squared_error  # ways to score the models

# ---------- 1. Rebuild the Week 1 features ----------
df = pd.read_csv("insurance.csv")  # load data
y = df['charges']  # target
X = pd.get_dummies(df.drop(columns='charges'), drop_first=True).astype(float)  # dummies for categorical columns
X['smoker_bmi'] = X['smoker_yes'] * df['bmi']  # interaction 1
X['smoker_age'] = X['smoker_yes'] * df['age']  # interaction 2 (let lasso decide if it matters!)
X['age_sq'] = df['age']**2  # polynomial term

# ---------- 2. Train/test split ----------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 80% train, 20% test

# ---------- 3. Scale the features ----------
scaler = StandardScaler()  # create the scaler
X_train_s = scaler.fit_transform(X_train)  # learn the scale from training data, then apply it
X_test_s = scaler.transform(X_test)  # apply the same scale to test data

# ---------- 4. Fit the models ----------
ols = LinearRegression().fit(X_train_s, y_train)  # plain regression, no penalty (for comparison)
ridge = RidgeCV(alphas=np.logspace(-2, 3, 50)).fit(X_train_s, y_train)  # ridge, tries 50 alphas and picks the best
lasso = LassoCV(cv=5, random_state=42, max_iter=10000).fit(X_train_s, y_train)  # lasso, picks best alpha using 5-fold CV
enet = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9], cv=5, random_state=42, max_iter=10000).fit(X_train_s, y_train)  # elastic net, also picks the ridge/lasso mix

print("Best alpha - Ridge:", ridge.alpha_)  # chosen penalty strength
print("Best alpha - Lasso:", lasso.alpha_)
print("Best alpha - Elastic Net:", enet.alpha_, "| l1_ratio:", enet.l1_ratio_)  # l1_ratio: 1 = all lasso, 0 = all ridge

# ---------- 5. Compare performance ----------
models = {'OLS': ols, 'Ridge': ridge, 'Lasso': lasso, 'Elastic Net': enet}  # put models in a dictionary
rows = []  # will hold one row of results per model
for name, m in models.items():  # loop through each model
    pred = m.predict(X_test_s)  # predict charges for the test set
    rows.append({
        'Model': name,
        'Train R2': m.score(X_train_s, y_train),  # fit on data it has seen
        'Test R2': r2_score(y_test, pred),  # fit on data it has NOT seen
        'Test RMSE': np.sqrt(mean_squared_error(y_test, pred))  # average prediction error in dollars
    })
results = pd.DataFrame(rows).round(4)  # turn into a table
print("\nMODEL COMPARISON")
print(results)

# ---------- 6. Compare coefficients ----------
coefs = pd.DataFrame({
    'OLS': ols.coef_, 'Ridge': ridge.coef_,
    'Lasso': lasso.coef_, 'Elastic Net': enet.coef_
}, index=X.columns).round(1)  # one column per model, one row per feature
print("\nCOEFFICIENTS (scaled features)")
print(coefs)  # look for zeros in the Lasso / Elastic Net columns

In [ ]:
# ---------- 1. Rebuild the Week 1 features ----------
df = pd.read_csv("insurance.csv")  # load data
y = df['charges']  # target
X = pd.get_dummies(df.drop(columns='charges'), drop_first=True).astype(float)  # dummies for categorical columns
X['smoker_bmi'] = X['smoker_yes'] * df['bmi']  # interaction 1
X['smoker_age'] = X['smoker_yes'] * df['age']  # interaction 2 (let lasso decide if it matters!)
X['age_sq'] = df['age']**2  # polynomial term